## Using GEDI Data

In [ ]:
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt

# 1. Open GEDI02A File
file_path = 'GEDI02_A_YYYYDDDHHMMSS_O_XX_XXX_XX_T00000_02_002_02.h5'
gedi_file = h5py.File(file_path, 'r')

# 2. Extract Data (Example for Beam BEAM0101)
beam = 'BEAM0101'
lats = gedi_file[f'{beam}/lat_lowestmode'][:]
lons = gedi_file[f'{beam}/lon_lowestmode'][:]
quality = gedi_file[f'{beam}/quality_flag'][:]
sensitivity = gedi_file[f'{beam}/sensitivity'][:]
elev_lowest = gedi_file[f'{beam}/elev_lowestmode'][:]

# RH98 is the 98th percentile of return energy (standard for canopy height)
rh98 = gedi_file[f'{beam}/rh'][:, 98] 

# 3. Create DataFrame and Filter
df = pd.DataFrame({
    'latitude': lats,
    'longitude': lons,
    'quality_flag': quality,
    'sensitivity': sensitivity,
    'elev_lowestmode': elev_lowest,
    'rh98': rh98
})

# Filter: Quality flag == 1 (high quality), Sensitivity > 0.9, and positive heights
filtered_df = df[
    (df['quality_flag'] == 1) & 
    (df['sensitivity'] > 0.9) & 
    (df['rh98'] > 0)
].dropna()

# 4. Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(filtered_df['longitude'], filtered_df['latitude'])]
gdf = gpd.GeoDataFrame(filtered_df, crs="EPSG:4326", geometry=geometry)

# 5. Quick Visualization
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(column='rh98', ax=ax, legend=True, cmap='viridis', markersize=5,
         legend_kwds={'label': "Canopy Height (m)"})
plt.title('GEDI Canopy Height (rh98) Transect')
plt.show()

gedi_file.close()


## Option 1: Pandas Coordinate Masking
### This approach integrates directly into the previous extraction script, filtering out unneeded global or orbit data before you convert the data to a spatial object.

In [ ]:
# Pandas Coordinate Masking

# Define your study area bounding box
min_lon, min_lat, max_lon, max_lat = -79.1721, 37.3296, -77.6873, 38.4755

# Apply coordinate mask alongside quality flags
spatial_and_quality_mask = (
    (df['longitude'] >= min_lon) & (df['longitude'] <= max_lon) &
    (df['latitude'] >= min_lat) & (df['latitude'] <= max_lat) &
    (df['quality_flag'] == 1) & 
    (df['sensitivity'] > 0.9) & 
    (df['rh98'] > 0)
)

filtered_df = df[spatial_and_quality_mask].dropna()


## Option 2: GeoPandas Bounding Box Clipping
### If you want to use advanced spatial tools, you can read the entire orbital transect into a GeoDataFrame and use the .clip() function.

In [ ]:
from shapely.geometry import box

# Create a bounding box polygon
study_area_box = box(min_lon, min_lat, max_lon, max_lat)

# Filter out points using GeoPandas clip
# (Assumes 'gdf' is your unfiltered GeoDataFrame with CRS EPSG:4326)
gdf_clipped = gpd.clip(gdf, study_area_box)


## Visualizing on a Map
### Because your study area covers a recognizable portion of Virginia (including parts of Charlottesville, Lynchburg, and Culpeper), adding a basemap is crucial for visual context. You can use contextily to automatically fetch web maps (like OpenStreetMap or Stamen terrain) underneath your GEDI tracks.

In [ ]:
import contextily as cx

# Contextily requires Web Mercator (EPSG:3857) to align with web tiles
gdf_clipped_3857 = gdf_clipped.to_crs(epsg=3857)

fig, ax = plt.subplots(figsize=(12, 10))

# Plot GEDI canopy tracks
gdf_clipped_3857.plot(
    column='rh98', 
    ax=ax, 
    legend=True, 
    cmap='YlGn',  # Green color ramp suited for trees
    markersize=8,
    alpha=0.7,
    legend_kwds={'label': "Canopy Height (rh98) in Meters"}
)

# Add basemap (OpenStreetMap Mapnik tiles)
cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)

ax.set_title('GEDI Tree Canopy Height - Virginia Study Area', fontsize=14)
ax.axis('off') # Remove axis ticks for a cleaner map
plt.show()


#### To create multi-year, year-over-year (YoY) comparisons between your GEDI canopy height data and the SMAP L3 Soil Moisture (SPL3SMP_E) data, you need to address a structural mismatch: SMAP provides a regular 9km grid, while GEDI gives sparse, narrow orbital footprints.

#### To visualizes them together cleanly, the best approach is to bin both datasets into matching spatial grids across your Virginia study area (-79.1721, 37.3296, -77.6873, 38.4755).

##### Step 1: Extract the Missing GEDI Year Attribute

#### The GEDI h5 files do not have a single "Year" variable. You must parse the collection year directly from the standard NASA Earthdata file names or use the delta_time timestamp.

In [ ]:
# For 390 files, parsing the file name string using Python's re module is computationally instant:

import re

# Standard GEDI naming: GEDI02_A_2022143... (YYYYDDD)
file_name = "GEDI02_A_2022143123456_O19345_02_T05643_02_002_01_V002.h5"

# Match 4 digits following the second underscore
year_match = re.search(r'GEDI02_A_(\d{4})', file_name)
if year_match:
    year = int(year_match.group(1)) # Returns: 2022


#### Step 3: Dual-Axis Year-Over-Year Visualizations

##### With both datasets gridded to the same resolution, you can create two types of powerful YoY visualizations:Visual 1: Spatial Grid Comparison (Side-by-Side Facet Plot)This plots maps of your Virginia study area across multiple years, directly comparing how spatial variations in soil moisture relate to structural changes in canopy height.

In [ ]:
years_to_plot = [2020, 2022, 2024]
fig, axes = plt.subplots(len(years_to_plot), 2, figsize=(14, 4 * len(years_to_plot)), sharex=True, sharey=True)

for idx, yr in enumerate(years_to_plot):
    # Left Column: GEDI Canopy Height
    ax_gedi = axes[idx, 0]
    # Filter xarray or dataframe for 'yr' and plot
    im1 = ax_gedi.imshow(ds_gedi['rh98'].sel(year=yr), extent=(min_lon, max_lon, min_lat, max_lat), cmap='YlGn')
    ax_gedi.set_title(f"{yr} GEDI Mean Canopy Height (rh98)")
    if idx == 0: fig.colorbar(im1, ax=ax_gedi, orientation='vertical')
        
    # Right Column: SMAP Soil Moisture
    ax_smap = axes[idx, 1]
    # Plot matching SMAP NetCDF array slice for 'yr'
    # im2 = ax_smap.imshow(smap_data_yr, extent=(min_lon, max_lon, min_lat, max_lat), cmap='Blues')
    ax_smap.set_title(f"{yr} SMAP Soil Moisture (SPL3SMP_E)")
    
plt.tight_layout()
plt.show()


#### Visual 2: Eco-Hydrological Trend Line (Regional Summary)

##### To see if severe soil moisture drops (droughts) precede canopy degradation or defoliation across the area over time, plot the regional yearly averages together on a dual y-axis chart.

In [ ]:
# Aggregate the entire study area into one average value per year
annual_canopy = df_gedi.groupby('year')['rh98'].mean()
# annual_soil_moisture = df_smap.groupby('year')['soil_moisture'].mean()

fig, ax1 = plt.subplots(figsize=(10, 5))

# Primary Axis: Canopy Height
color = 'tab:green'
ax1.set_xlabel('Year')
ax1.set_ylabel('Mean Region Canopy Height (m)', color=color)
ax1.plot(annual_canopy.index, annual_canopy.values, color=color, marker='o', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

# Secondary Axis: Soil Moisture
ax2 = ax1.twinx()  
color = 'tab:blue'
ax2.set_ylabel('Volumetric Soil Moisture ($m^3/m^3$)', color=color)
# ax2.plot(annual_soil_moisture.index, annual_soil_moisture.values, color=color, marker='s', linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Virginia Study Area: Year-Over-Year Ecological Trend')
plt.show()


#### Since I have successfully aggregated SMAP data by county/jurisdiction and year, the most efficient path forward is to aggregate the Downloaded raw GEDI point data to match this exact structure.  By grouping your GEDI spatial points into county boundaries, you avoid complex grid conversions and can directly merge the two datasets for your year-over-year (YoY) analysis.

#### Step 2: Merge GEDI with smap_df

##### Now that both datasets share the identical index keys (year and county), you can execute a clean Pandas merge.

In [ ]:
# Assuming your existing DataFrame is named `smap_df`
# Ensure 'year' data types match before merging (e.g., both int64)
gedi_county_summary['year'] = gedi_county_summary['year'].astype(int)
smap_df['year'] = smap_df['year'].astype(int)

# Combine datasets
merged_df = pd.merge(smap_df, gedi_county_summary, on=['year', 'county'], how='inner')


#### Step 3: Year-over-Year Visualizations

##### Visual 1: County-Level Multi-Panel Facet Plot (Trend Analysis)This layout creates an individual panel for each of your 8 jurisdictions, tracking how average soil moisture changes interact with canopy heights across the 2015–2023 time series.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up a grid of plots for the jurisdictions
g = sns.FacetGrid(merged_df, col="county", col_wrap=4, height=3.5, sharey=False)

def plot_dual_trend(data, **kwargs):
    ax1 = plt.gca()
    # Sort data by year to ensure correct line rendering
    data = data.sort_values('year')
    
    # Axis 1: Soil Moisture (Blue)
    color = '#1f77b4'
    ax1.plot(data['year'], data['sm_mean_m3m3'], color=color, marker='o', label='Soil Moisture')
    ax1.set_ylabel('Soil Moisture ($m^3/m^3$)', color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    
    # Axis 2: Canopy Height (Green)
    ax2 = ax1.twinx()
    color = '#2ca02c'
    ax2.plot(data['year'], data['canopy_height_mean_m'], color=color, marker='s', linestyle='--', label='Canopy Height')
    ax2.set_ylabel('Canopy Height (m)', color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    
    ax1.set_xlabel('Year')
    ax1.set_xticks(data['year'].unique())
    ax1.set_xticklabels(data['year'].unique(), rotation=45)

g.map_dataframe(plot_dual_trend)
g.set_titles("{col_name}")
plt.tight_layout()
plt.show()


#### Visual 2: Eco-Hydrological Scatter Correlation Plot

##### If you want to evaluate if regional canopy height structural density is actively correlated to soil moisture baselines, use a scatter baseline grouped by county.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=merged_df, 
    x='sm_mean_m3m3', 
    y='canopy_height_mean_m', 
    hue='county', 
    style='year', 
    s=100
)

plt.title('Relationship Between Surface Soil Moisture and Canopy Height (2015-2023)')
plt.xlabel('Volumetric Surface Soil Moisture ($m^3/m^3$)')
plt.ylabel('Mean Canopy Height rh98 (meters)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


#### To explore how past hydrological stress impacts future forest structure, you can implement a temporal lag using the Pandas .shift() function. Trees often exhibit a delayed physiological response to environmental changes; for instance, a severe root-zone drought in one year may result in canopy dieback, leaf-area reduction, or structural branch loss that only becomes measurable by GEDI satellite tracks 1 to 2 years later.

#### Step 1: Sorting and Shifting Data by County

##### To accurately lag the data, your dataset must be sorted chronologically within each individual county. If you simply shift the entire DataFrame, records will bleed across different county lines, corrupting the results.The script below applies a 1-year and 2-year lag to the SMAP soil moisture metrics using a Pandas groupby transformation:

In [ ]:
# 1. Sort values sequentially by county and year to ensure clean sliding shifts
merged_df = merged_df.sort_values(['county', 'year']).reset_index(drop=True)

# 2. Create a 1-Year Lag: Shift SMAP data forward by 1 row within each county group
# This matches LAST YEAR's Soil Moisture with THIS YEAR's Canopy Height
merged_df['sm_mean_lag1'] = merged_df.groupby('county')['sm_mean_m3m3'].shift(1)

# 3. Create a 2-Year Lag: Shift SMAP data forward by 2 rows
merged_df['sm_mean_lag2'] = merged_df.groupby('county')['sm_mean_m3m3'].shift(2)

# Optional: Inspect a slice to confirm the shift stayed within county boundaries
print(merged_df[['year', 'county', 'sm_mean_m3m3', 'sm_mean_lag1', 'canopy_height_mean_m']].head(5))


#### Step 2: Visualizing Lagged Cross-Correlations

##### To determine which time delay shows the strongest environmental relationship, you can plot multiple correlation lines simultaneously using a Seaborn regression visualization (lmplot). This helps you visually identify whether canopy changes track best with current soil moisture, a 1-year delay, or a 2-year delay.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Reshape data to long format for clean multi-variable plotting with Seaborn
melted_df = merged_df.melt(
    id_vars=['year', 'county', 'canopy_height_mean_m'],
    value_vars=['sm_mean_m3m3', 'sm_mean_lag1', 'sm_mean_lag2'],
    var_name='Lag_Type',
    value_name='Soil_Moisture_Value'
)

# Map human-readable labels to the lag metrics
lag_labels = {
    'sm_mean_m3m3': 'Same-Year Interaction (No Lag)',
    'sm_mean_lag1': '1-Year Hydrological Lag',
    'sm_mean_lag2': '2-Year Hydrological Lag'
}
melted_df['Lag_Type'] = melted_df['Lag_Type'].map(lag_labels)

# Drop any rows where shifting introduced NaN values (e.g., years 2015/2016 for lag metrics)
melted_df = melted_df.dropna(subset=['Soil_Moisture_Value', 'canopy_height_mean_m'])

# Create a multi-panel plot comparing the different delays
g = sns.lmplot(
    data=melted_df,
    x='Soil_Moisture_Value',
    y='canopy_height_mean_m',
    col='Lag_Type',
    hue='county',
    palette='Set2',
    height=5,
    aspect=1.1,
    scatter_kws={'s': 60, 'alpha': 0.8},
    robust=True # Handles outliers caused by sparse GEDI orbital coverage
)

g.set_axis_labels("Volumetric Surface Soil Moisture ($m^3/m^3$)", "Mean Canopy Height rh98 (m)")
g.set_titles("{col_name}")
plt.subplots_adjust(top=0.82)
g.fig.suptitle("Virginia Study Footprint: Assessing Canopy Height Response to Delayed Soil Moisture Stresses", fontsize=14)
plt.show()


#### Step 3: Quantifying the Best Lag Interval (Pearson r)

##### While visualizations point to visual trends, calculating the exact Pearson Correlation Coefficient (r) for each lag will statistically confirm which timeline dominant forces use to interact.

print("--- Regional Pearson Correlation Analysis ---")
for col_name, label in lag_labels.items():
    # Filter out NaNs to prevent statistical skewing
    valid_slice = merged_df[['canopy_height_mean_m', col_name]].dropna()
    
    if len(valid_slice) > 3:
        correlation = valid_slice['canopy_height_mean_m'].corr(valid_slice[col_name])
        print(f"{label:<32} : r = {correlation:.3f}")


#### Key Analytical Patterns to Watch ForPositive Correlation (r > 0.4): Higher soil moisture translates into healthier, taller, or denser forest structural profiles.The Peak Lag: Whichever lag configuration registers the highest absolute r-value indicates your forest system's specific structural buffer window to local climate fluctuations.